# Анализ логов бота Unpaywall

Этот ноутбук используется для анализа логов и подготовки статистики для диплома.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

# Настройки графиков
plt.style.use('ggplot')
%matplotlib inline

In [ ]:
# Загрузка логов
log_dir = Path('../data/logs')
logs = []

for log_file in sorted(log_dir.glob('access_*.jsonl')):
    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            logs.append(json.loads(line))

df = pd.DataFrame(logs)
print(f'Загружено {len(df)} записей')
df.head()

In [ ]:
# Преобразование timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

In [ ]:
# Статистика по дням
daily_counts = df.groupby('date').size()

plt.figure(figsize=(12, 6))
daily_counts.plot(kind='bar')
plt.title('Количество запросов по дням')
plt.xlabel('Дата')
plt.ylabel('Запросы')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Успешность по типам paywall
paywall_df = df[df['paywall'].notna()].copy()
paywall_df['paywall_type'] = paywall_df['paywall'].apply(lambda x: x.get('type'))

success_by_type = paywall_df.groupby('paywall_type')['status'].apply(
    lambda x: (x == 'success').mean() * 100
)

success_by_type.plot(kind='bar')
plt.title('Процент успешных обходов по типу paywall')
plt.ylabel('Успешность (%)')
plt.xlabel('Тип paywall')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Время ответа
success_df = df[df['status'] == 'success']

plt.figure(figsize=(10, 6))
plt.hist(success_df['duration_ms'], bins=50, alpha=0.7)
plt.title('Распределение времени ответа')
plt.xlabel('Время (мс)')
plt.ylabel('Количество запросов')
plt.show()

print(f'Среднее: {success_df["duration_ms"].mean():.0f} мс')
print(f'Медиана: {success_df["duration_ms"].median():.0f} мс')
print(f'95-й перцентиль: {success_df["duration_ms"].quantile(0.95):.0f} мс')

In [ ]:
# Топ пользователей
user_counts = df['user_id'].value_counts().head(10)

user_counts.plot(kind='bar')
plt.title('Топ-10 самых активных пользователей')
plt.xlabel('User ID')
plt.ylabel('Количество запросов')
plt.tight_layout()
plt.show()

In [ ]:
# Экспорт статистики для диплома
stats = {
    'total_requests': len(df),
    'unique_users': df['user_id'].nunique(),
    'success_rate': (df['status'] == 'success').mean() * 100,
    'avg_response_time': df[df['status'] == 'success']['duration_ms'].mean(),
    'total_articles': df['article'].notna().sum() if 'article' in df.columns else 0,
}

print('\n📊 Статистика для диплома:')
for key, value in stats.items():
    print(f'{key}: {value:.2f}' if isinstance(value, float) else f'{key}: {value}')